In [ ]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.cross_decomposition import PLSRegression
import matplotlib.pyplot as plt
import logging

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)
data.head()

## PCA降维

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

pred_y_list = []
true_y_list = []

logging.basicConfig(filename='PCR_result.log', level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.info('begin')


# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    #进行PCA降维
    pca=PCA(n_components='mle')
    X_train_after=pca.fit_transform(X_train,y=None)
    X_test_after=pca.transform(X_test)

    
    #使用降维后的数据进行线性回归
    model = LinearRegression(fit_intercept=False)
    model.fit(X_train_after, y_train)
    
    #without h
    y_test_pred = model.predict(X_test_after)

    
    # residuals = np.abs(model.predict(X_train_after) - y_train)

    # # 计算残差的99.9%分位数
    # epsilon = np.percentile(residuals, 99.9)
    
    # # 根据计算得到的epsilon值重新拟合模型
    # model1 = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon, \
    #                      penalty='elasticnet', alpha=0.0001, l1_ratio=0.6)
    # model1.fit(X_train_after, y_train)
    
    # # Predict on training and validation data
    # y_test_pred = model1.predict(X_test_after)
    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)
    
a='PCR(without h):(n_components='
b=np.array2string(pca.n_components_)
c='):'
d=np.array2string(R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100)
logging.info(a+b+c+d)
logging.info('end')

In [ ]:
print("PRC降维后R^2为：",R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100)

## PLS降维

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')


logging.info('begin')

pred_y_list = []
true_y_list = []



# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')

    # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    
 
    #进行PLS降维
        ##寻找最优参数
    
    # param_grid={'n_components': np.arange(1, X_train.shape[1] + 1)}
    # find_pls=PLSRegression()
    # grid_search=GridSearchCV(find_pls,param_grid,cv=2,scoring='r2',n_jobs=-1)
    # grid_search.fit(X_train,y_train)
    # best_n_components=grid_search.best_params_['n_components']

    n_component=44 

    
    # pls=PLSRegression(n_components= n_component)
    # pls.fit(X_train,y_train)
    # X_train_after = pls.transform(X_train)
    # X_test_after = pls.transform(X_test)


    # #使用降维后的数据进行线性回归（使用X_train）
    # residuals = np.abs(pls.predict(X_train) - y_train)

    # # 计算残差的99.9%分位数
    # epsilon = np.percentile(residuals, 99.9)
    
    # # 根据计算得到的epsilon值重新拟合模型
    # model = SGDRegressor(fit_intercept=False, loss='huber', epsilon=epsilon, \
    #                      penalty='elasticnet', alpha=0.0001, l1_ratio=0.5)
    # model.fit(X_train_after, y_train)
    
    # # Predict on training and validation data
    # y_test_pred = model.predict(X_test_after)

    
    #不加H

    pls=PLSRegression(n_components=n_component)
    pls.fit(X_train,y_train)
    y_test_pred=pls.predict(X_test)

    
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)


a='PLS(without h):(n_components='
b=str(n_component)
c='):'
d=np.array2string(R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100)
logging.info(a+b+c+d)
logging.info('end')

In [ ]:
print("PLR降维后R^2为：",R2_cal(np.array(true_y_list), np.array(pred_y_list))* 100)